# Preparing a liquid-solid interface

## Before you start

Either run `source ...` before opening the notebook, or define a variable for the GROMACS bin:

```gm_bin = "<path-to-gmx-bin>"```

and pass it to the `topology_compiler` functions.

Import utility libraries and set the variable for the work directory.

**NB** run the `workdir` cell only once!

In [ ]:
import sys  
sys.path.insert(1, '../libraries/')

In [ ]:
from topology_compiler import *
from droplet_spreading import *

In [ ]:
workdir = os.getcwd()
print("Work directory:",workdir)

## Topology generation

Unfortunately, `x2top` is basically useless, at least at the stage of generating the topology files for the solvant. If you already have `.itp` files for the solvant, move to the next step.

In [ ]:
help(test_gromacs_availability)

In [ ]:
test_gromacs_availability()

In [ ]:
help(run_x2top)

In [ ]:
run_x2top("HFO-1234zeE.gro", "refrigerants.ff", flags="-alldih -v")
# run_x2top("1233zd-trans.gro", "refrigerants.ff", flags="-alldih")

In [ ]:
!cat HFO-1234zeE.top
# !cat 1233zd-trans.top

In [ ]:
help(create_itp)

In [ ]:
create_itp("HFO-1234zeE.top", charge_list_file="charges-HFO-1234zeE.txt")

## Creating the liquiq-solid system

In [ ]:
!ls

In [ ]:
help(run_insert_molecules)

The number of molecules to insert needs to be guessed. This is not really a big deal, as it should attain its "physical" value after equilibration.

In [ ]:
n_added_mol = run_insert_molecules("zirconia.gro", "HFO-1234zeE.gro", flags="-try 10 -scale 0.65", nmol=1800)

In [ ]:
# Make sure it matches the output of 'run_insert_molecules'
print(n_added_mol)

In [ ]:
# Divide the number of atoms by 3 (ZrO2)
# 4800->1600
!cat zirconia.gro

In [ ]:
help(compile_topology)

In [ ]:
compile_topology(n_added_mol, 1600, "refrigerants.ff/forcefield.itp", "HFO-1234zeE.itp", "zirconia-header.txt")

In [ ]:
!cat topology-biphase.top

In [ ]:
!vmd zirconia-HFO-1234zeE.gro

In [ ]:
!tail zirconia-HFO-1234zeE.gro

The function `carve_gro` prints the total number of new atoms. This may be useful when compiling the topology file.

In [ ]:
zlow = 3.0
zupp = 14.55090-3.0
carve_condition = lambda x, y, z : (z>zlow)*(z<zupp)
carve_gro("zirconia-HFO-1234zeE.gro",9,carve_condition,mol_type='UNK',output_file="zirconia-HFO-1234zeE-carved.gro")

In [ ]:
!gmx editconf -f zirconia-HFO-1234zeE-carved.gro -o zirconia-HFO-1234zeE-carved.pdb

In [ ]:
!cat zirconia-HFO-1234zeE-carved.gro

In [ ]:
!vmd zirconia-HFO-1234zeE-carved.gro

# Molecular dynamics

In [ ]:
%cd {workdir}/molecular-dynamics/
!ls

**TODO**: Check how `topology-triphase.top` is created, the includes are not properly defined...

In [ ]:
!gmx grompp -p ../topology-triphase.top -c ../zirconia-HFO-1234zeE-carved.gro -r ../zirconia-HFO-1234zeE-carved.gro -f sd.mdp -o system-sd.tpr

In [ ]:
!gmx mdrun -v -s system-sd.tpr -deffnm steep

In [ ]:
!ls

In [ ]:
!gmx grompp -p ../topology-triphase.top -c steep.gro -r ../zirconia-HFO-1234zeE-carved.gro -f nvt.mdp -o system-nvt.tpr

In [ ]:
!gmx mdrun -v -s system-nvt.tpr -deffnm nvt

In [ ]:
!ls

In [ ]:
!vmd nvt.trr nvt.gro

In [ ]:
!gmx grompp -p ../topology-triphase.top -c nvt.gro -r ../zirconia-HFO-1234zeE-carved.gro -f npt.mdp -o system-npt.tpr

In [ ]:
!gmx mdrun -v -s system-npt.tpr -deffnm npt

In [ ]:
!ls

In [ ]:
!vmd npt.xtc npt.gro

# Test with hexane

In [ ]:
%cd {workdir}

In [ ]:
n_added_mol = run_insert_molecules("zirconia.gro", "hexane.gro", flags="-try 10 -scale 0.65", nmol=1000)

In [ ]:
compile_topology(n_added_mol, 1600, "refrigerants.ff/forcefield.itp", "hexane.itp", "zirconia-header.txt")

In [ ]:
%cd {workdir}/molecular-dynamics/
!ls

In [ ]:
!gmx grompp -p ../topology-biphase.top -c ../zirconia-hexane.gro -r ../zirconia-hexane.gro -f sd.mdp -o system-sd.tpr

In [ ]:
!gmx mdrun -v -s system-sd.tpr -deffnm steep

In [ ]:
!gmx grompp -p ../topology-biphase.top -c steep.gro -r ../zirconia-hexane.gro -f nvt.mdp -o system-nvt.tpr

In [ ]:
!gmx mdrun -v -s system-nvt.tpr -deffnm nvt

In [ ]:
!gmx grompp -p ../topology-biphase.top -c nvt.gro -r ../zirconia-hexane.gro -f npt.mdp -o system-npt.tpr

In [ ]:
!gmx mdrun -v -s system-npt.tpr -deffnm npt

In [ ]:
!vmd npt.xtc npt.gro